# Offline Analytics Template

Use this template for reproducible analysis from ClickHouse, Kafka replay topics, or S3/Parquet snapshots. Do not place secrets in notebooks; load them from environment variables or a local YAML file outside version control.

In [ ]:
from analytics.config import load_config
from analytics.clients.clickhouse import ClickHouseClient
from analytics.reports import write_report

config = load_config()  # or load_config("../config.local.yaml")
client = ClickHouseClient(config.clickhouse, config.query_safety)

In [ ]:
sql = """
SELECT block_timestamp, tx_hash, from_address, to_address, amount_usd, amount_usd_value
FROM telemetry_fabric.chain_token_transfers FINAL
WHERE tenant_id = {tenant_id:String}
  AND chain = {chain:String}
  AND network = {network:String}
  AND block_timestamp >= now64(3, 'UTC') - toIntervalDay({lookback_days:UInt32})
"""

params = {"tenant_id": "", "chain": "ethereum", "network": "mainnet", "lookback_days": 7}
df = client.query_dataframe(sql, params, limit=10_000)
df.head()

In [ ]:
# For larger reads, stream bounded chunks instead of loading all rows.
for chunk in client.query_dataframe_chunks(sql, params, total_limit=50_000, chunk_size=10_000):
    print(len(chunk))

In [ ]:
# Persist deterministic outputs with a manifest.
# write_report(df.to_dict(orient="records"), "reports/out/example.jsonl", metadata=params)